In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [ ]:
data = pd.read_csv('Churn_Modelling.csv')
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)
le=LabelEncoder()
data['Gender']=le.fit_transform(data['Gender'])
data.head()

In [ ]:
from sklearn.preprocessing import OneHotEncoder
oh=OneHotEncoder()
oh_df=oh.fit_transform(data[['Geography']])
col=oh.get_feature_names_out(['Geography'])
oh_df=oh_df.toarray()
oh_df=pd.DataFrame(oh_df,columns=col)
oh_df
data=pd.concat([data.drop('Geography',axis=1),oh_df],axis=1)

In [ ]:
data
with open('encoder_gender.pkl','wb') as file:
    pickle.dump(le,file)
with open('encoder_geography.pkl','wb') as file:
    pickle.dump(oh,file) 

In [ ]:
x=data.drop('Exited',axis=1)
y=data['Exited']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.20,random_state=42)
sc=StandardScaler()
x_train=sc.fit_transform(x_train)
x_test=sc.transform(x_test)
with open('scaler.pkl','wb') as file:
    pickle.dump(sc,file) 

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import datetime


In [ ]:
model=Sequential([
    Dense(64,activation='relu',input_shape=(x_train.shape[1],)),
    Dense(32,activation='relu'),
    Dense(1,activation='sigmoid')
]
)

In [ ]:
opt=tf.keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer="adam", loss="binary_crossentropy",metrics=['accuracy'])
loss=tf.keras.losses.BinaryCrossentropy()
loss

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
log_dir='logs/fit' + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensarflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)
Early_stoppping_callback=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)


In [ ]:
#training model
history=model.fit(
    x_train,y_train,validation_data=(x_test,y_test),epochs=100,
    callbacks=[tensarflow_callback,Early_stoppping_callback]
)

In [ ]:
model.save('model.h5')

In [ ]:
%load_ext tensorboard

In [ ]:
import os

log_dir = 'logs/fit20250613-232044'
print("Log dir exists:", os.path.exists(log_dir))
print("Subfolders:", os.listdir(log_dir))
for sub in os.listdir(log_dir):
    print(f"{sub} contents:", os.listdir(os.path.join(log_dir, sub)))


In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir logs